# Session 1: genes in pairs and the Punnett square
**A. Real-world problem.** Two parents each carry a pair of versions (alleles) of a gene.
A child gets one allele from each parent. If both parents show the same trait, can a child
still show a different one? A Punnett square lists every equally likely combination so
you can count.

*General education about inheritance in a simple textbook model. It is not medical or
genetic advice about any person or family.*

**Plan.** (1) alleles, genotype and appearance, (2) the Punnett square, (3) count the
ratios, (4) check the counts with a seeded simulation.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: genotype and appearance
Write the dominant allele as a capital letter (A) and the recessive allele as a small letter
(a). A parent's *genotype* is a pair: AA, Aa or aa. The *phenotype* (appearance) is
"dominant" if at least one A is present, and "recessive" only for aa.

In [ ]:
def gametes(genotype):
    """The two alleles a parent can pass on, e.g. 'Aa' -> ['A', 'a'] (each passed with chance 1/2)."""
    return list(genotype)


def child_genotype(a1, a2):
    """Order the pair so that the capital letter comes first: ('a', 'A') -> 'Aa'."""
    return "".join(sorted([a1, a2]))


def punnett(parent1, parent2):
    """2 x 2 table of child genotypes: rows = alleles of parent 1, columns = alleles of parent 2."""
    return [[child_genotype(x, y) for y in gametes(parent2)] for x in gametes(parent1)]


def phenotype(genotype):
    return "dominant" if "A" in genotype else "recessive"

## Step 2: count the four boxes
Every box of the square is equally likely, so counting boxes gives the ratios.
For Aa x Aa we expect genotypes AA : Aa : aa = 1 : 2 : 1 and appearance 3 : 1.

In [ ]:
def genotype_counts(parent1, parent2):
    counts = {"AA": 0, "Aa": 0, "aa": 0}
    for row in punnett(parent1, parent2):
        for g in row:
            counts[g] += 1
    return counts


def phenotype_counts(parent1, parent2):
    counts = {"dominant": 0, "recessive": 0}
    for g, n in genotype_counts(parent1, parent2).items():
        counts[phenotype(g)] += n
    return counts

## Step 3: simulate many children
Each child gets one random allele from each parent. A seeded generator repeats exactly.
The counts will be close to, but not exactly, 3 : 1.

In [ ]:
def simulate_children(parent1, parent2, n, seed=1):
    """Return the number of children with the dominant and the recessive appearance."""
    rng = np.random.default_rng(seed)
    a1 = np.array(gametes(parent1))[rng.integers(0, 2, size=n)]
    a2 = np.array(gametes(parent2))[rng.integers(0, 2, size=n)]
    recessive = int(np.sum((a1 == "a") & (a2 == "a")))
    return {"dominant": n - recessive, "recessive": recessive}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    for ax, (p1, p2) in zip(axes, [("Aa", "Aa"), ("AA", "aa"), ("Aa", "aa")]):
        table = punnett(p1, p2)
        for i in range(2):
            for j in range(2):
                g = table[i][j]
                ax.add_patch(plt.Rectangle((j, 1 - i), 1, 1, fc="#9ecae1" if phenotype(g) == "dominant" else "#fdae6b", ec="k"))
                ax.text(j + 0.5, 1.5 - i, g, ha="center", va="center", fontsize=16)
            ax.text(-0.15, 1.5 - i, gametes(p1)[i], ha="center", va="center", fontsize=14)
            ax.text(i + 0.5, 2.15, gametes(p2)[i], ha="center", va="center", fontsize=14)
        gc, pc = genotype_counts(p1, p2), phenotype_counts(p1, p2)
        ax.set_title(f"{p1} x {p2}\nAA:Aa:aa = {gc['AA']}:{gc['Aa']}:{gc['aa']}   dominant:recessive = {pc['dominant']}:{pc['recessive']}", fontsize=9)
        ax.set_xlim(-0.4, 2.1)
        ax.set_ylim(-0.1, 2.5)
        ax.set_aspect("equal")
        ax.axis("off")
        print(p1, "x", p2, "genotypes", gc, "appearance", pc)
    fig.suptitle("Punnett squares (blue = dominant appearance, orange = recessive appearance)", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT / "punnett_squares.png", dpi=120)

    sizes = [10, 100, 1000, 10000]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    fracs = []
    for n in sizes:
        r = simulate_children("Aa", "Aa", n)
        fracs.append(r["recessive"] / n)
        print(f"{n:6d} children: dominant {r['dominant']}, recessive {r['recessive']}, fraction recessive {fracs[-1]:.3f}")
    ax2.semilogx(sizes, fracs, "o-", label="simulated fraction")
    ax2.axhline(0.25, color="r", ls="--", label="Punnett square: 1/4")
    ax2.set_xlabel("number of children simulated")
    ax2.set_ylabel("fraction with the recessive appearance")
    ax2.set_title("Aa x Aa: counts approach 1/4 as the family grows")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "recessive_fraction.png", dpi=120)

## Where this stops
Real traits can have incomplete dominance, many genes and environment effects. This is the
one-gene starting point. Session 2 adds the probability rules and whole populations.

## Coding tasks
1. **Easy.** Change the cross in `lesson.py` to Aa x aa and print the counts. Predict them first.
2. **Stretch.** Write `chance_recessive(parent1, parent2)` that returns the probability of an aa child from the Punnett square and check it for all nine pairings.
3. **Explain.** Explain, with the square, why two parents who both show the dominant trait can have a child who shows the recessive trait but two aa parents cannot.

## Assessment
- **A. Conceptual.** Two parents both show the dominant trait and have a child with the recessive trait. What must the genotypes of the parents be? Why?
- **B. Numerical.** For the cross AA x Aa, fill in the Punnett square and give the genotype counts and the chance that a child shows the recessive appearance.
- **C. Graph and simulation interpretation.** In the second figure, why does the fraction of recessive children jump around for 10 children but sit close to 1/4 for 10000?
- **D. Design and optimisation challenge.** Design: you cross two plants and want about half of the offspring to show the recessive appearance. Which parent genotypes would you choose, and why not the other choices?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    gc = L.genotype_counts("Aa", "Aa")
    pc = L.phenotype_counts("Aa", "Aa")
    checks.append(("worked example: Aa x Aa gives genotypes AA:Aa:aa = 1:2:1", (gc["AA"], gc["Aa"], gc["aa"]) == (1, 2, 1)))
    checks.append(("worked example: Aa x Aa gives appearance 3 dominant : 1 recessive", (pc["dominant"], pc["recessive"]) == (3, 1)))
    gc2 = L.genotype_counts("AA", "aa")
    checks.append(("AA x aa: all four boxes are Aa", gc2 == {"AA": 0, "Aa": 4, "aa": 0}))
    pc3 = L.phenotype_counts("Aa", "aa")
    checks.append(("Aa x aa gives appearance 2 : 2 (a 1 : 1 ratio)", (pc3["dominant"], pc3["recessive"]) == (2, 2)))
    checks.append(("aa x aa: all four boxes are aa", L.genotype_counts("aa", "aa") == {"AA": 0, "Aa": 0, "aa": 4}))
    checks.append(("the four boxes always add up to 4", all(sum(L.genotype_counts(a, b).values()) == 4 for a in ("AA", "Aa", "aa") for b in ("AA", "Aa", "aa"))))
    checks.append(("order of letters does not matter: child_genotype('a','A') is 'Aa'", L.child_genotype("a", "A") == "Aa"))
    r = L.simulate_children("Aa", "Aa", 10000)
    checks.append((f"simulation of 10000 Aa x Aa children: recessive fraction near 0.25 (got {r['recessive'] / 10000:.3f})", abs(r["recessive"] / 10000 - 0.25) < 0.02))
    r0 = L.simulate_children("AA", "Aa", 2000)
    checks.append(("AA x Aa gives genotypes AA:Aa:aa = 2:2:0", L.genotype_counts("AA", "Aa") == {"AA": 2, "Aa": 2, "aa": 0}))
    checks.append(("AA x Aa never gives a recessive child", r0["recessive"] == 0))
    checks.append(("a seeded simulation repeats exactly", L.simulate_children("Aa", "Aa", 500, seed=3) == L.simulate_children("Aa", "Aa", 500, seed=3)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
